# gguf-serve — any GGUF model, one public OpenAI-compatible API

Run cell 1, then cell 2. Cell 1 clones the repo; cell 2 does everything else: installs
the CUDA build of llama.cpp, downloads and verifies the model, loads it across your
GPUs, and serves a chat UI plus an OpenAI-compatible API on one public URL.

**Everything you can change is a named constant at the top of cell 2**, already set to
its shipped default — the model, context length, GPU split, and port. Edit a value in
place and re-run the cell; nothing else needs touching.

**Before you start**, set the accelerator to **GPU T4 x2**
(Kaggle: *Settings -> Accelerator*). The default model, Qwen3.8-27B at Q5_K_XL, needs
about 23 GiB of VRAM — that is two T4s.

On free Colab you only get a single 16 GB T4 and the default will not fit. Either use an
L4 or A100 runtime, or in cell 2 set:

```
MODEL_FILE = "Qwen3.8-27B-UD-Q3_K_XL.gguf"
TENSOR_SPLIT = "none"
```

First run takes roughly 15 minutes, most of it downloading the model.

In [ ]:
# 1 — Get the code
#
# Change REPO_URL if you forked the repo.

REPO_URL = "https://github.com/kishorsharma/gguf-serve.git"

import os
import subprocess
import urllib.error
import urllib.request
from pathlib import Path

# Without this, an HTTPS URL that GitHub will not serve anonymously does not
# fail — it asks for a username and this cell hangs forever waiting for one.
# No username would help, so failing immediately is strictly better.
os.environ["GIT_TERMINAL_PROMPT"] = "0"

workdir = next(
    (p for p in (Path("/kaggle/working"), Path("/content")) if p.is_dir()),
    Path.cwd(),
)
repo = workdir / "gguf-serve"

_done = 0
_total = 2 if (repo / ".git").is_dir() else 3


def step(text):
    global _done
    _done += 1
    print(f"\n[{_done}/{_total}] {text}", flush=True)


def run(*cmd, cwd=None):
    print("   $", " ".join(str(c) for c in cmd), flush=True)
    return subprocess.run(cmd, cwd=cwd).returncode


def anonymous_clone_works(url):
    """Ask GitHub whether it will hand this repo to an anonymous client.

    Cheap pre-flight: it turns the credential prompt, which reads as a hang,
    into one line of explanation before we ever invoke git.
    """
    probe = url.removesuffix(".git") + ".git/info/refs?service=git-upload-pack"
    try:
        with urllib.request.urlopen(probe, timeout=30) as response:
            return response.status == 200, f"HTTP {response.status}"
    except urllib.error.HTTPError as error:
        return False, f"HTTP {error.code}"
    except Exception as error:  # DNS, TLS, no network
        return False, f"{type(error).__name__}: {error}"


if (repo / ".git").is_dir():
    step(f"{repo} already exists, updating it")
    if run("git", "pull", "--ff-only", "--progress", cwd=repo) == 0:
        print("   [ok] up to date")
    else:
        print("   [!] update failed — continuing with the copy already on disk")
else:
    step(f"Checking {REPO_URL}")
    reachable, detail = anonymous_clone_works(REPO_URL)
    if not reachable:
        print(f"   [x] GitHub will not serve that repo anonymously ({detail}).")
        print("       Usually one of:")
        print("       - it has not been pushed yet: create the repo on GitHub,")
        print("         then `git push -u origin main` from your machine")
        print("       - the owner or name in REPO_URL above is a typo")
        print("       - it is private, so an anonymous clone cannot see it. Use")
        print("         https://<token>@github.com/<owner>/<repo>.git instead")
        raise SystemExit("cannot clone REPO_URL")
    print(f"   [ok] reachable ({detail})")

    step(f"Cloning into {repo}")
    if run("git", "clone", "--depth", "1", "--progress", REPO_URL, str(repo)) != 0:
        raise SystemExit("git clone failed — see the output above")
    print("   [ok] cloned")

step("Entering the repo")
os.chdir(repo)
print("   [ok] working directory is", Path.cwd())

In [ ]:
# 2 — Settings, then install, download, load and serve
#
# Every value below is the shipped default, so this cell runs as-is. Change any
# of them in place and re-run; nothing else needs editing. Sampling is set per
# request by the client, so it is not here.
#
# Leave the cell running: the server lives inside it, so stopping the cell takes
# the public URL down with it. Watch for the https://....gradio.live line — that
# is your public URL.

# --- which model ------------------------------------------------------------
# A repo and one file inside it.
MODEL_REPO = "unsloth/Qwen3.8-27B-GGUF"
MODEL_FILE = "Qwen3.8-27B-UD-Q5_K_XL.gguf"

# Or paste the address bar of any .gguf page and ignore the two lines above.
MODEL_URL = ""   # e.g. "https://huggingface.co/unsloth/Qwen3-8B-GGUF/blob/main/Qwen3-8B-Q4_K_M.gguf"

# Where the download lands. /tmp is roomy but wiped when the runtime restarts.
MODEL_DIR = "/tmp/gguf-serve/models"

# --- how it runs ------------------------------------------------------------
# Context window in tokens. This model accepts up to 262144; past about 131072
# set KV_CACHE to "q8_0" as well, or the cache will not fit.
CTX = 16384

# "f16" is full precision. "q8_0" halves what the context costs in VRAM for a
# small quality cost, which is what makes the longest contexts possible at all.
KV_CACHE = "f16"

# -1 puts every layer on the GPU. Lower it to keep some on the CPU when a model
# is slightly too big — much slower, but it runs.
GPU_LAYERS = -1

# How to divide the model across GPUs. "1,1" is an even split for a matched
# pair, "1,2" weights it for 16 GB plus 32 GB, "none" is a single GPU.
TENSOR_SPLIT = "1,1"

# --- server -----------------------------------------------------------------
PORT = 7860

# False serves only inside this notebook, with no public URL.
SHARE = True

# Reasoning models close their scratchpad with </think>. True splits that off so
# clients get clean answers; False passes the output through untouched.
PARSE_REASONING = True

# ----------------------------------------------------------------------------
import shlex
import subprocess
import sys

command = [
    sys.executable, "launch.py",
    "--model-dir", MODEL_DIR,
    "--ctx", str(CTX),
    "--kv-cache-type", KV_CACHE,
    "--gpu-layers", str(GPU_LAYERS),
    "--tensor-split", TENSOR_SPLIT,
    "--port", str(PORT),
]

# A URL names the repo and the file by itself, so passing both would let them
# contradict each other.
if MODEL_URL.strip():
    command += ["--model", MODEL_URL.strip()]
else:
    command += ["--model-repo", MODEL_REPO, "--model-file", MODEL_FILE]

if not SHARE:
    command += ["--no-share"]
if not PARSE_REASONING:
    command += ["--no-reasoning"]

print("running:", " ".join(shlex.quote(part) for part in command), flush=True)
subprocess.run(command)

## Using the API from anywhere

Once the public URL is up, any OpenAI client works against it. Run this from your
laptop, not from this notebook:

```python
from openai import OpenAI

client = OpenAI(
    base_url="https://YOUR-ID.gradio.live/v1",
    api_key="not-used",  # this server does not check keys
)

response = client.chat.completions.create(
    model="qwen3.8-27b-ud-q5-k-xl",
    messages=[{"role": "user", "content": "Hello!"}],
)

print(response.choices[0].message.content)
```

`GET /health` reports the exact model id if you changed the model. The URL is
temporary: it is gone as soon as this notebook stops.

## Notes

- **The model is downloaded to `/tmp` and lost on restart.** `/kaggle/working` is
  capped at 20 GB, which is too small. To keep it, add the GGUF as a Kaggle dataset
  and set `MODEL_DIR = "/kaggle/input/<your-dataset>"` in cell 2.
- **Anyone with the public link can use the model.** There is no authentication.
  Set `SHARE = False` in cell 2 if you only want access inside this notebook.
- The constants in cell 2 cover everything most people need. For the few remaining
  settings, and the reasoning behind the defaults, see `ggufserve/config.py` and
  `docs/configuration.md`.